#### 01 - Bronze Layer: Chicago Food Inspections
**Medallion Architecture - Bronze Zone**

Raw ingestion from source CSV. No transformations applied — load as-is into Delta table.

##### Setup: Database and Paths

In [0]:
# Config
CHICAGO_SOURCE = "/Volumes/workspace/default/damg7370/food_inspections/raw/Food_Inspections_20260406.csv"
BRONZE_DB      = "bronze"
BRONZE_TABLE   = "bronze_chicago_inspections"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")
print(f"Database ready: {BRONZE_DB}")

##### Step 1: Read Raw CSV (no transforms)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(CHICAGO_SOURCE)
)

print(f"Row count   : {df_raw.count():,}")
print(f"Column count: {len(df_raw.columns)}")
df_raw.printSchema()

##### Step 2: Add Ingestion Metadata

In [0]:
import re

def clean_col_name(name):
    name = name.lower().strip()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    return name

df_renamed = df_raw
for col_name in df_raw.columns:
    new_name = clean_col_name(col_name)
    df_renamed = df_renamed.withColumnRenamed(col_name, new_name)

print("Renamed columns:")
for c in df_renamed.columns:
    print(f"  {c}")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_renamed
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file",         lit(CHICAGO_SOURCE))
    .withColumn("_source_system",       lit("Chicago_OpenData"))
    .withColumn("_city",                lit("Chicago"))
)

print("Metadata columns added")
display(df_bronze.limit(3))

##### Step 3: Write to Bronze Delta Table

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_DB}.{BRONZE_TABLE}")
)

print(f"✅ Written to: {BRONZE_DB}.{BRONZE_TABLE}")

##### Step 4: Verify

In [0]:
df_verify = spark.table(f"{BRONZE_DB}.{BRONZE_TABLE}")
row_count = df_verify.count()

print(f"✅ Row count : {row_count}")
print(f"✅ Columns   : {len(df_verify.columns)}")
assert row_count > 300000, f"Expected ~308161 rows, got {row_count}"
print("✅ Assertion passed")
display(df_verify.limit(5))